<a href="https://colab.research.google.com/github/shreyaspathak11/SLM/blob/main/notebooks/train_text_to_sql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Fine-Tuning a Text-to-SQL Small Language Model (SLM) from Scratch!
### *A Fun, Friendly, Step-by-Step Guide to Turning a 3B Model into an Expert SQL Engineer — Explained for a 5-Year-Old!*
*(Companion to the Vizuara AI Labs SLM from Scratch & Antigravity Full-Stack AI Engineering Series)*

---

### 🌟 What is Text-to-SQL and Why Does It Matter?
Imagine you own a bustling toy store with thousands of teddy bears, LEGO sets, and video games. All your sales records are locked inside a giant, mysterious steel filing cabinet called a **Relational Database (SQL)**! 🗄️
* **The Human Boss:** Speaks friendly everyday English: *"Hey, show me the top 5 most expensive toys sold in Europe this month!"* 🗣️
* **The Database Filing Cabinet:** Only understands strict, unforgiving computer commands: `SELECT name, price FROM Products WHERE ...` 🤖
* **Our Mission:** We will train our clever **Puppy Robot (Llama-3.2-3B or Phi-3-Mini)** to become the world's fastest bilingual translator between human speech and database queries! ✨

---

### 🗺️ The 3 Big Secrets of Full-Stack AI Engineering:
1. **The Fuel (Data):** 🥣 78,000+ real-world database blueprints from the **`b-mc2/sql-create-context`** dataset. Every example shows the question, the database table schema, and the exact SQL query!
2. **The Magic Wand (4-Bit QLoRA):** 🪄 Fine-tuning 3 Billion parameters normally requires a giant $10,000 GPU and 16+ GB of VRAM. With **Quantization (NF4)** and **LoRA (Low-Rank Adapters)**, we freeze the brain, attach tiny post-it notes, and train it on a **100% free Google Colab T4 GPU** using only ~2.5 GB of memory!
3. **The Pocket Rocket (GGUF & llama.cpp):** 🚀 Instead of keeping slow, giant PyTorch files, we merge the weights and export to a quantized **GGUF** binary file. It can be downloaded to your laptop and run lightning-fast on ordinary CPU or GPU without needing PyTorch!

Let's build, train, test, and export our SQL specialist step by step! 🚀


## 🛠️ Step 1: Check GPU & Install Libraries (Setting Up the AI Workshop!)

### 🏎️ Why do we use "Unsloth"? (The Racecar Engine Analogy)
Imagine you want to drive across the country:
* **Standard PyTorch & Hugging Face:** Like driving a reliable family station wagon. It gets you there, but it uses lots of fuel (VRAM) and takes 10 hours.
* **Unsloth:** Like swapping in a custom-built Formula-1 rocket engine! 🏎️⚡
  * The Unsloth team wrote custom **OpenAI Triton GPU kernels** from scratch.
  * It cuts memory usage by **70%** and makes fine-tuning **2x to 5x faster** without losing any accuracy!
  * This is what allows us to train a cutting-edge 3B model right inside Google Colab's free tier without crashing!


In [ ]:
# ==============================================================================
# 🛠️ STEP 1: VERIFY GPU & INSTALL UNSLOTH + HUGGING FACE TOOLS
# ==============================================================================
# 🧒 ELI5 Analogy:
# We are opening the garage door, checking our GPU engine with 'nvidia-smi',
# and unpacking our specialized power tools: Unsloth, TRL, PEFT, and Datasets!

# 1. Verify that Colab gave us a GPU (Look for Tesla T4 or A100)
!nvidia-smi

# 2. Install Unsloth and essential fine-tuning packages
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes datasets

## 📖 Step 2: Load the Dataset (The Database Blueprint Library!)

### 🕵️ Why do we need "Schema Context"? (The Detective Analogy)
Imagine a detective walks into a crime scene and asks: *"Who stole the diamonds?"*
* **Without Context:** The detective doesn't know who was in the house, what rooms exist, or what keys belong to what doors. They hallucinate wild guesses: *"Maybe the butler named Bob did it!"* (Even if there is no butler in the house!).
* **With Context (Database Schema):** You hand the detective a blueprint of the mansion: `CREATE TABLE Rooms (id INT, name TEXT);` and `CREATE TABLE Guests (id INT, room_id INT);`. Now the detective can write an exact, flawless investigation query!

The dataset **`b-mc2/sql-create-context`** provides **78,577 examples** containing:
1. **`question`**: Human query (*"How many head coaches are older than 45?"*)
2. **`context`**: The exact table DDL schema (*`CREATE TABLE table_name (head_coach TEXT, age INTEGER);`*)
3. **`answer`**: The target SQL statement (*`SELECT COUNT(*) FROM table_name WHERE age > 45;`*)


In [ ]:
# ==============================================================================
# 📖 STEP 2: LOAD B-MC2/SQL-CREATE-CONTEXT FROM HUGGING FACE
# ==============================================================================
# 🧒 ELI5 Analogy:
# Walking into Hugging Face's digital library and checking out 78,000 real-world
# database puzzles so our robot student can practice reading schemas!

from datasets import load_dataset

# Download the training split
dataset = load_dataset("b-mc2/sql-create-context", split="train")

print(f"🎉 Dataset loaded successfully!")
print(f"Total training examples: {len(dataset):,}")
print("\n--- Sneak Peek at Example #0 ---")
print("Question:", dataset[0]["question"])
print("Context: ", dataset[0]["context"])
print("Answer:  ", dataset[0]["answer"])

## 🍱 Step 3: The Alpaca Prompt Format (The 3-Compartment Bento Box!)

### 🍱 Why Prompt Formatting Matters:
If you toss rice, chicken curry, and fresh strawberries into a blender together, you get a muddy mess! 🍓🍛🤢
Instead, you serve your meal in a **Bento Box with 3 clean compartments**:

```text
┌────────────────────────────────────────────────────────────────────────┐
│ 1. ### Instruction: What is the task? (e.g. Write a SQL query...)      │
├────────────────────────────────────────────────────────────────────────┤
│ 2. ### Input: What is the background context? (The CREATE TABLE DDL)   │
├────────────────────────────────────────────────────────────────────────┤
│ 3. ### Response: What is the desired answer? (The SELECT statement)    │
└────────────────────────────────────────────────────────────────────────┘
```

By training the model on this exact structure:
* During inference, we fill compartments 1 & 2, write `### Response:\n`, and stop!
* The model instantly knows its turn has arrived and outputs pure, clean SQL without chatty conversational filler.


In [ ]:
# ==============================================================================
# 🍱 STEP 3: DEFINE THE ALPACA TEMPLATE & FORMAT THE DATASET
# ==============================================================================
# 🧒 ELI5 Analogy:
# Stamping every single training flashcard with our official 3-compartment bento box template
# and gluing the secret '<|end_of_text|>' (EOS) sticker so the robot knows when to stop talking!

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Write a SQL query to answer this question: {question}

### Input:
{context}

### Response:
{answer}"""

# Define the formatting function
def formatting_prompts_func(examples, eos_token="<|end_of_text|>"):
    questions = examples["question"]
    contexts = examples["context"]
    answers = examples["answer"]
    texts = []
    for q, c, a in zip(questions, contexts, answers):
        # Format prompt and append the end-of-sequence token
        text = alpaca_prompt.format(question=q, context=c, answer=a) + eos_token
        texts.append(text)
    return {"text": texts}

print("Template defined! Ready to apply to our base model tokenizer.")

## 🎒 Step 4: Load Base Model in 4-Bit Precision (The Superhero in a Backpack!)

### ⚖️ The Precision Scale: FP16 vs NF4 Quantization
* **16-bit Float (FP16):** Like carrying 3 Billion solid gold bricks. It's accurate down to 7 decimal places, but it weighs **6 to 14 GB** in memory and crashes our GPU.
* **4-bit NormalFloat (NF4):** Like converting those gold bricks into high-tech digital cards! 💳
  * Each number is squashed into just **4 bits** (storing numbers from 0 to 15 mapped along a bell curve).
  * Memory drops to **~1.8 GB**!
  * Amazingly, language models retain **99%+ of their reasoning power** even when quantized to 4 bits!


In [ ]:
# ==============================================================================
# 🎒 STEP 4: LOAD LLAMA-3.2-3B-INSTRUCT IN 4-BIT PRECISION
# ==============================================================================
# 🧒 ELI5 Analogy:
# Waking up our friendly 3-Billion-parameter robot puppy and fitting its brain
# into a lightweight 4-bit travel backpack so it comfortably fits inside our Colab GPU!

import torch
from unsloth import FastLanguageModel

max_seq_length = 2048   # Maximum reading window for schema + question + SQL
dtype = None            # Auto-detects Float16 on T4, Bfloat16 on newer GPUs
load_in_4bit = True     # Compress weights into 4-bit NF4 to save 70% VRAM

# Load model and tokenizer via Unsloth's optimized loader
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print("🎉 Llama-3.2-3B loaded successfully in 4-bit precision!")
print(f"Tokenizer EOS Token: {tokenizer.eos_token} (ID: {tokenizer.eos_token_id})")

## 📝 Step 5: Configure LoRA (The Post-It Note Apprenticeship!)

### 🧒 Why LoRA? (The 3-Billion Weight Problem)
Imagine you hire a brilliant chef who already knows how to speak, chop vegetables, and operate an oven:
* **Full Fine-Tuning (Brain Surgery):** You open up the chef's skull and adjust all 3 Billion neurons. It takes months, risks brain damage (catastrophic forgetting), and costs thousands of dollars!
* **LoRA (Low-Rank Adaptation):** You hand the chef a small pad of **Post-It notes**! 📝
  * The chef's original brain weights $W$ are permanently **FROZEN**.
  * We only train two tiny matrices $A$ and $B$ that represent SQL-specific tips:
    $$\Delta W = \frac{\alpha}{r} (B \cdot A)$$
  * With rank $r=16$, trainable parameters drop from **3,200,000,000 down to ~20,000,000** (< 0.6% of the model)!


In [ ]:
# ==============================================================================
# 📝 STEP 5: ATTACH LORA ADAPTERS (RANK=16, ALPHA=32)
# ==============================================================================
# 🧒 ELI5 Analogy:
# Freezing all 3 Billion base weights and pinning lightweight SQL flashcard adapters
# onto the attention flashlights (q_proj, k_proj, v_proj, o_proj) and feed-forward layers!

model = FastLanguageModel.get_peft_model(
    model,
    r=16,                   # Rank: width of the low-rank bottleneck
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha=32,          # Scaling factor (standard rule of thumb is 2 * r)
    lora_dropout=0,         # 0 for Unsloth-optimized kernels
    bias="none",
    use_gradient_checkpointing="unsloth", # Saves 50%+ memory during backprop
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

# Inspect how few parameters we need to train!
model.print_trainable_parameters()

## 🏋️ Step 6: Supervised Training with SFTTrainer (The SQL Gym!)

### 🛒 The Training Gear:
* **`per_device_train_batch_size = 2`**: Our GPU holds 2 examples per bite.
* **`gradient_accumulation_steps = 4`**: Like making 4 trips with our shopping cart before checking out ($2 \times 4 = 8$ effective batch size).
* **`learning_rate = 2e-4`**: A gentle, steady step size guided by the AdamW 8-bit optimizer.
* **`max_steps = 250`**: In Colab, 250 steps takes ~5-10 minutes and drops the loss from ~2.2 down to < 0.6!


In [ ]:
# ==============================================================================
# 🏋️ STEP 6: TRAIN OUR SQL SPECIALIST WITH SFTTRAINER
# ==============================================================================
# 🧒 ELI5 Analogy:
# Our robot puppy sits down at the SQL study desk and practices 250 rounds
# of translating human questions + schemas into flawless SQL queries!

from trl import SFTTrainer
from transformers import TrainingArguments

# 1. Apply our Alpaca formatting to the dataset
def apply_formatting(examples):
    return formatting_prompts_func(examples, eos_token=tokenizer.eos_token)

formatted_dataset = dataset.map(apply_formatting, batched=True)

# Select a high-quality subset for fast, focused training in Colab
train_sample = formatted_dataset.select(range(min(15000, len(formatted_dataset))))

# 2. Configure training arguments
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_sample,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=250,              # Adjust to 500-1000 for full convergence
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=25,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="text_to_sql_outputs",
    ),
)

# 3. Run training and watch the loss decrease smoothly!
print("🚀 Starting Supervised Fine-Tuning...")
train_stats = trainer.train()
print("🎉 Training completed successfully!")

## 🎯 Step 7: The Final Exam! (Testing on our Project's Schema)

Before exporting, let's test our newly educated model on the exact e-commerce schema from our **`analytics.db`** database:
- `Users (id, name, signup_date, region)`
- `Products (id, name, category, price)`
- `Orders (id, user_id, product_id, order_date, quantity)`


In [ ]:
# ==============================================================================
# 🎯 STEP 7: TEST INFERENCE ON A REAL PROJECT QUERY
# ==============================================================================
# 🧒 ELI5 Analogy:
# Handing our robot puppy a graduation test quiz with our exact store schema
# to see if it writes the correct SQL without peeking at the answers!

# Switch model to fast inference mode
FastLanguageModel.for_inference(model)

test_schema = """CREATE TABLE Users (id INTEGER PRIMARY KEY, name TEXT, signup_date TEXT, region TEXT);
CREATE TABLE Products (id INTEGER PRIMARY KEY, name TEXT, category TEXT, price REAL);
CREATE TABLE Orders (id INTEGER PRIMARY KEY, user_id INTEGER, product_id INTEGER, order_date TEXT, quantity INTEGER);"""

test_question = "What is the total quantity of products ordered by users in Europe?"

test_prompt = alpaca_prompt.format(
    question=test_question,
    context=test_schema,
    answer="" # Leave empty for the model to complete!
)

# Tokenize and generate response
inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")
outputs = model.generate(
    **inputs,
    max_new_tokens=128,
    use_cache=True,
    temperature=0.1, # Keep low for deterministic, syntactically correct SQL
)

generated_text = tokenizer.batch_decode(outputs)[0]
sql_response = generated_text.split("### Response:")[-1].replace(tokenizer.eos_token, "").strip()

print("🤖 Human Question:", test_question)
print("\n✨ Model Generated SQL:")
print(sql_response)

## 🚀 Step 8: Export to Quantized GGUF (`Q4_K_M`)

### 📦 Why GGUF? (The Portable Pocket Guide Analogy)
Now that our model is a trained expert, how do we run it on our laptop?
* **PyTorch Weights (.safetensors):** Requires a 15 GB Python virtual environment, CUDA drivers, and heavy memory.
* **GGUF (`llama.cpp` format):** Like packaging the model into a single, high-efficiency binary executable! 💾
  * Merges the LoRA post-it notes directly into the base weights.
  * Quantizes with **`Q4_K_M`** (medium 4-bit precision with high-precision attention layers).
  * Result: A single **~2.2 GB file** that runs at 25–40+ tokens per second on an ordinary CPU or laptop!


In [ ]:
# ==============================================================================
# 🚀 STEP 8: MERGE LORA & EXPORT AS GGUF (Q4_K_M)
# ==============================================================================
# 🧒 ELI5 Analogy:
# Freezing our chef's post-it notes permanently into the master recipe book,
# compressing it into a tiny pocket-sized edition ('llama-3.2-3b-sql-q4_k_m.gguf'),
# and getting it ready to send down to our Antigravity local project!

# Export to GGUF using Unsloth's automated conversion
model.save_pretrained_gguf(
    "llama-3.2-3b-sql-gguf",
    tokenizer,
    quantization_method="q4_k_m"
)

print("🎉 Export complete! Your GGUF model is ready in the 'llama-3.2-3b-sql-gguf' directory.")

## 🤝 Step 9: Download to Local Machine & Connect to Antigravity

Run the cell below to download your `.gguf` file to your computer.
Then, move it to your local project directory:
👉 **`d:/SLM/model/llama-3.2-3b-sql-q4_k_m.gguf`**

Once placed there, your Antigravity local backend (`backend/llm_inference.py` and `backend/main.py`) will automatically load the model for your Streamlit analytics dashboard! 🎈


In [ ]:
# ==============================================================================
# 🤝 STEP 9: DOWNLOAD THE GGUF FILE TO YOUR LOCAL COMPUTER
# ==============================================================================

import glob
from google.colab import files

gguf_files = glob.glob("llama-3.2-3b-sql-gguf/*.gguf")
if gguf_files:
    print(f"📦 Found exported model: {gguf_files[0]}")
    print("Starting browser download... Once downloaded, move it to d:/SLM/model/")
    files.download(gguf_files[0])
else:
    print("⚠️ No .gguf file found. Please check the export directory in Step 8.")